# Somo la 10 - Wakala wa AI katika Uzalishaji

Katika somo hili utajifunza **mifumo ya uzalishaji** kwa wawakala wa AI ukitumia Microsoft Agent Framework (Python). Tunahusika na:

- **Ufuatiliaji** — kuongeza muda na kurekodi kwa mwingiliano wa wakala
- **Tathmini** — kutumia wakala mtathmini kupima ubora wa majibu
- **Usimamizi wa gharama** — mikakati ya ufanisi wa tokeni na uchaguzi wa modeli

Muktadha ni **wakala wa usafiri** anayesaidia watumiaji kupanga safari, kwa ufuatiliaji na tathmini vilivyo juu yake.


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mambo ya Kuzingatia Katika Uzalishaji

Kuhamisha mawakala wa AI kutoka kwa majaribio hadi uzalishaji kunahitaji umakini makini kwa nguzo tatu:

1. **Uwezo wa Kuona** — Unahitaji kuona kile wakala anachofanya, kina muda gani, na ni zana zipi anazitumia. Bila kufuatilia na kurekodi, kutatua matatizo ya uzalishaji ni karibu haiwezekani.

2. **Tathmini** — Ukaguzi wa moja kwa moja wa ubora huhakikisha majibu ya wakala yanabaki sahihi, kamili, na ya msaada kwa muda. Wakala wa mtathmini anaweza kutoa alama kwa majibu kulingana na vigezo vilivyowekwa.

3. **Usimamizi wa Gharama** — Matumizi ya tokeni huathiri gharama moja kwa moja. Mikakati kama kuboresha maagizo, kuchagua modeli, na kuhifadhi husaidia kudhibiti matumizi bila kupoteza ubora.


## Kujenga Wakala Anayeweza Kuangaliwa

Tunafafanua zana za kusafiri na kuzunguka wito wa wakala na upimaji wa muda ili tuweze kufuatilia kuchelewa. Katika uzalishaji ungeunganisha na OpenTelemetry au mfumo mwingine wa ufuatiliaji.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mifumo ya Tathmini

Msimbo wa uzalishaji wa kawaida ni kutumia waki wa pili kama **mtathmini**. Mtathmini hutoa alama kwa jibu la wakala mkuu dhidi ya vigezo vilivyowekwa kama ukamilifu, usahihi, na msaada.

Hii inawezesha:
- Vizuizi vya ubora vya kiotomatiki kabla ya majibu kufika kwa watumiaji
- Kugundua kushuka kwa kiwango wakati maagizo au mifano yanabadilika
- Ufuatiliaji endelevu wa utendaji wa wakala kwa muda


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mikakati ya Usimamizi wa Gharama

Kudhibiti gharama ni muhimu kwa mawakala wa AI wa uzalishaji. Hapa kuna mikakati muhimu:

| Mkakati | Maelezo |
|---|---|
| **Uboreshaji wa maelekezo** | Hifadhi maagizo ya mfumo kuwa mafupi. Ondoa muktadha wa ziada ili kupunguza tokeni za maingizo. |
    "| **Chaguo la mfano** | Tumia mifano midogo, ya bei nafuu (mfano GPT-4.1-mini) kwa kazi rahisi kama ugawaji au uchimbaji, na uhifadhi mifano mikubwa kwa hoja ngumu. |\n",
| **Kuweka matokeo kivinjari** | Hifadhi matokeo ya zana na maswali yanayorudiwa ili kuepuka simu za API za ziada. |
| **Mikakati ya tokeni** | Weka mipaka ya `max_tokens` ili kuzuia majibu marefu yasiyotarajiwa. |
| **Kukusanya pamoja** | Kusanya maswali mengi ya mtumiaji katika simu moja ya API pale inavyowezekana. |

Kivitendo, njia ya ngazi nyingi hufanya kazi vizuri: elekeza maombi rahisi kwa mfano wa haraka na wa bei nafuu na ongeza maombi magumu tu kwa mfano wenye uwezo zaidi.


## Muhtasari

Katika somo hili umejifunza jinsi ya:

1. **Kuongeza ufuatiliaji** katika mwingiliano wa mawakala kwa kutumia upimaji wa wakati na kurekodi, kuanzisha msingi wa kufuatilia na usimamizi.
2. **Kutathmini majibu ya mawakala** moja kwa moja kwa kutumia wakala mtathmini anayepima ukamilifu, usahihi, na ufanisi.
3. **Kudhibiti gharama** kupitia uboreshaji wa maagizo, uchaguzi wa modeli, kuhifadhi taarifa, na bajeti za tokeni.

Mifumo hii ya uzalishaji husaidia kuhakikisha mawakala wako wa AI ni wa kuaminika, yanayopimika, na yenye gharama nafuu kwa kiwango kikubwa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
